# 3.4 调用接口开发

Kernel 开发完成后，需要提供对外接口供用户调用。本节介绍 aclnn 和 PTA 两种接口的开发方式。

---

## 学习目标

- 了解 aclnn 接口的自动生成和手动构建方式
- 理解 L0 / L2 的分层架构和 executor 的作用
- 掌握 PTA 调用 aclnn 的通用模式
- 了解内部接口和对外接口的区别

---

## 1. aclnn 接口

aclnn 是 CANN 提供的 C 语言算子接口，遵循**两段式**设计：

```
Phase 1: GetWorkspaceSize → 构建计算图，返回需要的 workspace 大小和执行器
Phase 2: Op            → 传入 workspace + executor，执行计算
```

aclnn 接口有两种构建方式：**自动生成** 和 **手动构建**。

### 1.1 CMake 自动生成

通过 `CMakeLists.txt` 中 `add_modules_sources` 的 `ACLNNTYPE` 参数控制：

```cmake
# fast_gelu 的 CMakeLists.txt
set(SUPPORT_COMPUTE_UNIT "ascend950")
set(SUPPORT_TILING_DIR "arch35")
add_modules_sources(HOSTNAME ${OPHOST_NAME}
    MODE PRIVATE
    COMPUTE_UNIT ${SUPPORT_COMPUTE_UNIT}
    TILING_DIR ${SUPPORT_TILING_DIR}
    OPTYPE fast_gelu
    ACLNNTYPE aclnn_exclude)
```

`ACLNNTYPE` 有三种取值，决定算子的 `_def.cpp` 被编译到哪个目标，从而控制接口的可见范围：

| ACLNNTYPE | 含义 | def.cpp 路由 | 使用场景 |
|------|------|------|------|
| `aclnn` | 对外公开接口 | `${OPHOST_NAME}_opdef_aclnn_obj` | 框架自动生成对外的 `aclnnXxx` 接口 |
| `aclnn_inner` | 内部接口 | `${OPHOST_NAME}_opdef_aclnn_inner_obj` | 仅 CANN 内部模块可调用的接口 |
| `aclnn_exclude` | 手动包装 | `${OPHOST_NAME}_opdef_aclnn_exclude_obj` | 不走自动生成，由开发者手动实现 `aclnn_*.h/.cpp` |

区别：
- **对外接口** (`aclnn`)：头文件暴露给用户，函数名需符合 `aclnnXxxGetWorkspaceSize` / `aclnnXxx` 命名规范
- **内部接口** (`aclnn_inner`)：不对用户暴露，仅 CANN 内部组件使用
- **手动包装** (`aclnn_exclude`)：跳过自动生成，开发者自己写 `op_api/aclnn_*.h` 和 `aclnn_*.cpp`

FastGelu 使用 `aclnn_exclude`，因为其接口需要特殊的参数校验和连续性处理，不方便由框架自动生成。

### 1.2 手动构建：两段式接口

手动构建时，开发者需要自己实现 `aclnn_*.h` 和 `aclnn_*.cpp`。以 FastGelu 为例——

**头文件** (`aclnn_fast_gelu.h`)：

```c
#include "aclnn/aclnn_base.h"

ACLNN_API aclnnStatus aclnnFastGeluGetWorkspaceSize(
    const aclTensor* self, aclTensor* out,
    uint64_t* workspaceSize, aclOpExecutor** executor);

ACLNN_API aclnnStatus aclnnFastGelu(
    void* workspace, uint64_t workspaceSize,
    aclOpExecutor* executor, aclrtStream stream);
```

**Phase 1 实现** — 构建计算图：

```cpp
aclnnStatus aclnnFastGeluGetWorkspaceSize(..., aclOpExecutor** executor) {
    // 1. 创建执行器
    auto uniqueExecutor = CREATE_EXECUTOR();

    // 2. 参数校验
    CheckParams(self, out);

    // 3. 处理非连续输入（如有需要）
    auto selfContiguous = l0op::Contiguous(self, uniqueExecutor.get());

    // 4. 调用 L0 算子 → 注册到执行器
    auto result = l0op::FastGelu(selfContiguous, uniqueExecutor.get());

    // 5. 处理非连续输出（如有需要）
    l0op::ViewCopy(result, out, uniqueExecutor.get());

    // 6. 获取 workspace 大小，交出执行器
    *workspaceSize = uniqueExecutor->GetWorkspaceSize();
    uniqueExecutor.ReleaseTo(executor);
    return ACLNN_SUCCESS;
}
```

**Phase 2 实现** — 执行计算图：

```cpp
aclnnStatus aclnnFastGelu(void* workspace, uint64_t workspaceSize,
                          aclOpExecutor* executor, aclrtStream stream) {
    return CommonOpExecutorRun(workspace, workspaceSize, executor, stream);
}
```

Phase 2 非常简洁，只是将 Phase 1 构建好的 executor 提交到 stream 上执行。两段式接口的整体调用流程如下：

<img src="./images/aclnn_two_phase_flow.png" alt="aclnn two phase flow" width="700px">

### 1.3 executor 的作用

`aclOpExecutor` 是一个**计算图容器**，在 Phase 1 中逐步构建、在 Phase 2 中一次性执行：

```
Phase 1 构建过程：
  executor = CREATE_EXECUTOR()          → 创建空执行器
  l0op::Contiguous(self, executor)       → 注册 Contiguous 操作
  l0op::FastGelu(self, executor)         → 注册 FastGelu 操作
  l0op::ViewCopy(result, out, executor) → 注册 ViewCopy 操作
  executor.GetWorkspaceSize()            → 计算整张图需要的 workspace

Phase 2 执行过程：
  CommonOpExecutorRun(workspace, executor, stream)
    → 按注册顺序依次执行 Contiguous → FastGelu → ViewCopy
```

executor 内部维护四种核心能力——内存管理（AllocTensor、Device/Host tensor）、计算图管理（每个 L0 算子对应一个 KernelLauncher 节点）、Workspace 计算（分析 tensor 生命周期做内存复用以压缩总大小）、Kernel 下发（按计算图顺序依次 Launch 到 stream）。

### 1.4 L2 与 L0 的分工

aclnn 调用链路分为两层：

```
用户调用
  │
  ├── aclnnFastGelu (L2 层 / aclnn)
  │     ├── 参数校验（CheckParams）
  │     ├── 连续性处理（Contiguous / ViewCopy）
  │     ├── Workspace 计算
  │     └── 调用 l0op::FastGelu ──────┐
  │                                   │
  ├── l0op::FastGelu (L0 层 / op executor)
  │     ├── AllocTensor：分配输出张量
  │     ├── ADD_TO_LAUNCHER_LIST_AICORE：注册 AICORE 启动
  │     └── 返回 aclTensor* 给 L2
  │
  └── AI Core Kernel（内核执行）
```

| 层级 | 文件 | 职责 |
|------|------|------|
| **L2 (aclnn)** | `aclnn_fast_gelu.h/.cpp` | 参数校验、连续性处理、workspace 管理、两段式接口 |
| **L0 (l0op)** | `fast_gelu.h/.cpp` | 张量分配、核函数注册到 executor |

L2 处理"算子的外壳"——校验、连续性、workspace；L0 处理"算子的内核"——注册到 AICORE 启动列表。L2 接口内部会组合调用多个 L0 算子来构建完整的计算图：

<img src="./images/aclnn_internal_framework.png" alt="aclnn internal framework" width="700px">

L0 算子采用**多 Kernel 实例化**架构——针对不同的输入组合（dtype + format）编译出独立的 `.o` 文件（如 `add_f32_nd.o`、`add_bf16_nd.o`）。运行时 nnopbase 框架通过 `KernelMgr → OpKernel → OpKernelBin` 三级组件管理这些二进制，根据入参生成 hash key 精确匹配，预加载到内存避免下发时的磁盘 I/O。

此外，CANN 中有两级 workspace：L0 Workspace 由 Tiling 函数计算（Kernel 临时空间），L2 Workspace 由 `GetWorkspaceSize()` 计算并做内存复用。Tiling 函数由每个 L0 算子注册，Phase 1 或 Phase 2 中执行，返回 tiling_key、blockDim 和 workspace_size。

---

## 2. PTA 接口

PTA（PyTorch Ascend）是 PyTorch 集成昇腾的接口，本质上是 **Python 包装 aclnn 的两段式 C API**。

<img src="./images/pta_architecture.png" alt="PTA architecture" width="700px">

当用户在 NPU 上调用 `torch.matmul(a, b)` 时，PyTorch 框架内部通过 Dispatcher 机制自动完成路由。C++ 层 `THPVariable_matmul` 进入 Dispatcher，从 tensor 中提取 dispatch key set（`{Autograd, PrivateUse1}`）。Autograd 层处理梯度后通过 `redispatch` 剥离 Autograd key，命中 `PrivateUse1` 下 torch_npu 注册的 kernel。torch_npu 完成设备校验后调用 `op_plugin::matmul` → aclnn 两段式接口。`PrivateUse1` 是 PyTorch 预留给第三方加速设备的 dispatch key，全过程对用户透明：

<img src="./images/dispatch.png" alt="PyTorch Dispatcher" width="700px">

### 2.1 通用实现逻辑

PTA 函数内部调用 aclnn 接口的通用流程：

```
Python: torch_npu.npu_fast_gelu(x)
  │
  ├── 1. 将 PyTorch Tensor 转换为 aclTensor (C 结构体)
  ├── 2. Phase 1: aclnnFastGeluGetWorkspaceSize → 获取 executor + workspaceSize
  ├── 3. 分配 workspace 内存
  ├── 4. Phase 2: aclnnFastGelu → 提交 executor 到 stream 执行
  ├── 5. SynchronizeStream 等待完成
  └── 6. 将结果 aclTensor 转换回 PyTorch Tensor
```

### 2.2 内部 torch_npu 接口 vs 对外接口

| 类别 | 接口形式 | 可见范围 | 示例 |
|------|------|------|------|
| **内部接口** | `torch_npu.npu_xxx` | 仅 CANN 内部使用，API 可能变更 | `torch_npu.npu_fast_gelu(x)` |
| **对外接口** | `torch.nn.functional.xxx` 或 `F.xxx` | 用户通过标准 PyTorch API 调用 | `F.gelu(x, approximate='tanh')` |

区别：
- **内部接口**注册在 `torch_npu` 模块下，命名带 `npu_` 前缀，直接映射到 aclnn
- **对外接口**通过 PyTorch 的 `__torch_dispatch__` 或 `torch.library` 机制注册，用户在不知道底层是 NPU 的情况下就能调用标准 PyTorch API

---

## 小结

| 知识点 | 要点 |
|------|------|
| aclnn 自动生成 | `ACLNNTYPE` 控制：`aclnn`(对外) / `aclnn_inner`(内部) / `aclnn_exclude`(手动) |
| aclnn 手动构建 | 两段式：Phase 1 构建 executor + 校验 + workspace；Phase 2 提交到 stream 执行 |
| executor | 计算图容器，Phase 1 注册操作，Phase 2 一次性执行；兼管内存、workspace 和 Kernel 下发 |
| L0 算子管理 | 多 Kernel 实例化（按 dtype/format 编译独立 .o 文件），hash 匹配后直接加载执行 |
| PTA | Python 包装 aclnn；通过 PyTorch Dispatcher 的 `PrivateUse1` 自动路由 NPU tensor |

---

上一节：[3.2 SIMD 算子开发](./03.02_simd_operator_development.ipynb)  
下一节：[3.4 算子性能优化](./03.04_performance_optimization.ipynb)